# 09 - Consultas Analiticas
## Analisis sobre el Modelo Estrella (Capa Oro)

---

### Objetivo de este notebook

Este notebook contiene **9 consultas analiticas** ejecutadas sobre el modelo
dimensional (star schema) de la capa Oro. Cada consulta responde a una
pregunta de negocio relevante sobre la bioactividad de los medicamentos.

### Tablas utilizadas

| Tabla | Tipo | Descripcion |
|-------|------|-------------|
| `pubchem_oro.fact_bioactividades` | Hechos | Metricas de bioactividad |
| `pubchem_oro.dim_compuestos` | Dimension | Medicamentos y propiedades |
| `pubchem_oro.dim_ensayos` | Dimension | Ensayos biologicos |
| `pubchem_oro.dim_resultados` | Dimension | Tipos de resultado |

### Preguntas que responden las consultas

1. Que medicamentos tienen mas bioensayos?
2. Cual es la proporcion de resultados activos vs inactivos?
3. Que compuestos tienen mayor tasa de actividad biologica?
4. Como se distribuyen los tipos de ensayo?
5. Cuales son los compuestos mas estudiados?
6. Que compuestos cumplen la Regla de Lipinski?
7. Cuales son los compuestos mas potentes (menor valor de actividad)?
8. Que ensayos tienen mayor tasa de resultados activos?
9. Cual es el perfil de actividad de cada medicamento?

---
## 1. Preparacion: Registrar Vistas Temporales

Para facilitar la escritura de consultas SQL, registramos las tablas
del modelo estrella como **vistas temporales** con nombres cortos.
Esto permite usar `fact`, `dim_comp`, `dim_ens` y `dim_res`
en lugar de los nombres completos de las tablas.

In [ ]:
# Registrar cada tabla del modelo estrella como vista temporal
# Las vistas temporales existen solo durante la sesion de Spark activa
spark.table("pubchem_oro.fact_bioactividades").createOrReplaceTempView("fact")       # Tabla de hechos
spark.table("pubchem_oro.dim_compuestos").createOrReplaceTempView("dim_comp")        # Dimension compuestos
spark.table("pubchem_oro.dim_ensayos").createOrReplaceTempView("dim_ens")            # Dimension ensayos
spark.table("pubchem_oro.dim_resultados").createOrReplaceTempView("dim_res")         # Dimension resultados

print("Vistas temporales registradas exitosamente.")
print("  fact     -> pubchem_oro.fact_bioactividades")
print("  dim_comp -> pubchem_oro.dim_compuestos")
print("  dim_ens  -> pubchem_oro.dim_ensayos")
print("  dim_res  -> pubchem_oro.dim_resultados")

---
## Consulta 1: Top 10 Compuestos con Mas Bioensayos

**Pregunta de negocio:** Cuales son los medicamentos mas estudiados?

Identificamos los 10 medicamentos con mayor cantidad de ensayos biologicos
registrados. Los compuestos mas estudiados suelen ser los de mayor
interes farmacologico o los que llevan mas tiempo en investigacion.

In [ ]:
# Consulta 1: Top 10 compuestos por cantidad de ensayos unicos
# Se cuenta tanto ensayos unicos como registros totales para diferenciar
# compuestos probados en muchos ensayos vs compuestos con muchas replicas
spark.sql("""
    SELECT
        dc.nombre_comun,                              -- Nombre del medicamento
        dc.id_compuesto,                              -- CID de PubChem
        COUNT(DISTINCT f.llave_ensayo) AS total_ensayos,  -- Ensayos unicos
        COUNT(*) AS total_registros                   -- Total de filas (incluye replicas)
    FROM fact f
    JOIN dim_comp dc ON f.llave_compuesto = dc.llave_compuesto
    GROUP BY dc.nombre_comun, dc.id_compuesto
    ORDER BY total_ensayos DESC
    LIMIT 10
""").show(truncate=False)

---
## Consulta 2: Distribucion de Resultados de Actividad

**Pregunta de negocio:** Cual es la proporcion global de resultados activos vs inactivos?

Analiza la distribucion de los diferentes tipos de resultado
(Active, Inactive, Inconclusive, Probe) en todo el dataset.
Esto da una vision general de la selectividad de los compuestos.

In [ ]:
# Consulta 2: Distribucion global de tipos de resultado
# Se usa una window function (SUM OVER) para calcular el porcentaje
# respecto al total sin necesidad de subconsulta
spark.sql("""
    SELECT
        dr.resultado_actividad,                       -- Tipo de resultado
        COUNT(*) AS cantidad,                         -- Cantidad de registros
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(),  -- Porcentaje del total
            2
        ) AS porcentaje
    FROM fact f
    JOIN dim_res dr ON f.llave_resultado = dr.llave_resultado
    GROUP BY dr.resultado_actividad
    ORDER BY cantidad DESC
""").show(truncate=False)

---
## Consulta 3: Compuestos con Mayor Tasa de Actividad

**Pregunta de negocio:** Que medicamentos muestran mayor actividad biologica?

Calcula para cada medicamento el porcentaje de ensayos donde fue clasificado
como "Active". Una tasa alta de actividad indica mayor versatilidad
farmacologica o mayor promiscuidad del compuesto.

In [ ]:
# Consulta 3: Tasa de actividad por compuesto
# Se usa CASE WHEN para contar condicionalmente los resultados 'Active'
spark.sql("""
    SELECT
        dc.nombre_comun,                              -- Nombre del medicamento
        COUNT(*) AS total_ensayos,                    -- Total de ensayos
        SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) AS ensayos_activos,
        ROUND(
            SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS tasa_actividad_pct                       -- Porcentaje de ensayos activos
    FROM fact f
    JOIN dim_comp dc ON f.llave_compuesto = dc.llave_compuesto
    JOIN dim_res dr ON f.llave_resultado = dr.llave_resultado
    GROUP BY dc.nombre_comun
    ORDER BY tasa_actividad_pct DESC
""").show(truncate=False)

---
## Consulta 4: Distribucion por Tipo de Ensayo

**Pregunta de negocio:** Que tipos de ensayos biologicos predominan en el dataset?

Analiza la composicion metodologica del dataset agrupando por tipo de ensayo.
Los tipos comunes incluyen Screening (cribado), Confirmatory (confirmacion)
y Summary (resumen).

In [ ]:
# Consulta 4: Distribucion por tipo de ensayo
# Se cuentan tanto ensayos unicos como registros totales por tipo
spark.sql("""
    SELECT
        de.tipo_ensayo,                               -- Tipo de ensayo biologico
        COUNT(DISTINCT de.id_ensayo) AS ensayos_unicos,  -- Ensayos distintos
        COUNT(*) AS total_registros,                  -- Total de registros
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(),  -- Porcentaje del total
            2
        ) AS porcentaje
    FROM fact f
    JOIN dim_ens de ON f.llave_ensayo = de.llave_ensayo
    GROUP BY de.tipo_ensayo
    ORDER BY total_registros DESC
""").show(truncate=False)

---
## Consulta 5: Compuestos Mas Probados

**Pregunta de negocio:** Cuales son los medicamentos con mayor volumen de datos?

Muestra los medicamentos con mayor cantidad total de registros de bioactividad,
incluyendo su clasificacion Lipinski y el promedio de actividad en micromolar.

In [ ]:
# Consulta 5: Compuestos mas probados con metricas detalladas
spark.sql("""
    SELECT
        dc.nombre_comun,                              -- Nombre del medicamento
        dc.peso_molecular,                            -- Peso molecular en Daltons
        dc.clasificacion_lipinski,                    -- Clasificacion de biodisponibilidad
        COUNT(*) AS total_registros,                  -- Total de registros
        COUNT(DISTINCT f.llave_ensayo) AS ensayos_distintos,  -- Ensayos unicos
        ROUND(AVG(f.valor_actividad_um), 4) AS promedio_actividad_um  -- Promedio de actividad
    FROM fact f
    JOIN dim_comp dc ON f.llave_compuesto = dc.llave_compuesto
    GROUP BY dc.nombre_comun, dc.peso_molecular, dc.clasificacion_lipinski
    ORDER BY total_registros DESC
""").show(truncate=False)

---
## Consulta 6: Compuestos que Cumplen la Regla de Lipinski

**Pregunta de negocio:** Cuales de nuestros medicamentos tienen buena biodisponibilidad oral?

Lista los compuestos que cumplen todos los criterios de la Regla de los 5 de Lipinski,
mostrando sus propiedades fisicoquimicas completas para verificacion.

In [ ]:
# Consulta 6: Detalle de compuestos que cumplen la Regla de Lipinski
# Se filtran directamente de la dimension de compuestos
spark.sql("""
    SELECT
        nombre_comun,                                 -- Nombre del medicamento
        id_compuesto,                                 -- CID de PubChem
        formula_molecular,                            -- Formula quimica
        peso_molecular,                               -- Peso molecular (debe ser <= 500)
        xlogp,                                        -- Lipofilicidad (debe ser <= 5)
        tpsa,                                         -- Area de superficie polar
        complejidad,                                  -- Complejidad molecular
        clasificacion_lipinski                        -- Confirmacion de clasificacion
    FROM dim_comp
    WHERE clasificacion_lipinski = 'Cumple Lipinski'
    ORDER BY nombre_comun
""").show(truncate=False)

---
## Consulta 7: Top Compuestos por Potencia Promedio

**Pregunta de negocio:** Cuales son los compuestos mas potentes?

En farmacologia, la potencia se mide inversamente al valor de actividad:
un **menor valor en micromolar** indica **mayor potencia** (se necesita
menos cantidad del compuesto para lograr el efecto).

Solo se consideran registros con resultado "Active" y valor de actividad disponible.

In [ ]:
# Consulta 7: Compuestos ordenados por potencia promedio
# Menor valor de actividad = mayor potencia farmacologica
spark.sql("""
    SELECT
        dc.nombre_comun,                              -- Nombre del medicamento
        COUNT(*) AS ensayos_activos_con_valor,        -- Ensayos activos con valor numerico
        ROUND(AVG(f.valor_actividad_um), 4) AS promedio_actividad_um,   -- Potencia promedio
        ROUND(MIN(f.valor_actividad_um), 4) AS minimo_actividad_um,     -- Mejor potencia
        ROUND(MAX(f.valor_actividad_um), 4) AS maximo_actividad_um      -- Menor potencia
    FROM fact f
    JOIN dim_comp dc ON f.llave_compuesto = dc.llave_compuesto
    JOIN dim_res dr ON f.llave_resultado = dr.llave_resultado
    WHERE dr.resultado_actividad = 'Active'           -- Solo ensayos activos
      AND f.valor_actividad_um IS NOT NULL             -- Solo con valor numerico
    GROUP BY dc.nombre_comun
    ORDER BY promedio_actividad_um ASC                 -- Menor valor = mayor potencia
""").show(truncate=False)

---
## Consulta 8: Ensayos con Mayor Tasa de Actividad

**Pregunta de negocio:** Que ensayos biologicos detectan mas actividad?

Identifica los ensayos con mayor proporcion de resultados activos.
Se aplica un filtro de **minimo 50 pruebas** para garantizar
significancia estadistica y evitar ensayos con muy pocos datos.

In [ ]:
# Consulta 8: Ensayos con mayor tasa de actividad (minimo 50 pruebas)
# La clausula HAVING filtra despues de la agregacion
spark.sql("""
    SELECT
        de.id_ensayo,                                 -- ID del ensayo
        de.nombre_ensayo,                             -- Nombre del ensayo
        de.tipo_ensayo,                               -- Tipo de ensayo
        COUNT(*) AS total_pruebas,                    -- Pruebas totales
        SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) AS pruebas_activas,
        ROUND(
            SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS tasa_actividad_pct                       -- Tasa de actividad
    FROM fact f
    JOIN dim_ens de ON f.llave_ensayo = de.llave_ensayo
    JOIN dim_res dr ON f.llave_resultado = dr.llave_resultado
    GROUP BY de.id_ensayo, de.nombre_ensayo, de.tipo_ensayo
    HAVING COUNT(*) >= 50                             -- Minimo 50 pruebas
    ORDER BY tasa_actividad_pct DESC
    LIMIT 15
""").show(truncate=False)

---
## Consulta 9: Perfil de Actividad por Compuesto

**Pregunta de negocio:** Cual es el comportamiento completo de cada medicamento?

Vista comparativa integral que muestra para cada medicamento:
- Cantidad de resultados activos, inactivos, inconclusos y otros
- Porcentaje de actividad e inactividad
- Clasificacion Lipinski

Esta es la consulta mas completa para entender el perfil farmacologico.

In [ ]:
# Consulta 9: Perfil completo de actividad por medicamento
# Desglosa todos los tipos de resultado para cada compuesto
spark.sql("""
    SELECT
        dc.nombre_comun,                              -- Nombre del medicamento
        dc.clasificacion_lipinski,                    -- Clasificacion Lipinski
        SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) AS activos,
        SUM(CASE WHEN dr.resultado_actividad = 'Inactive' THEN 1 ELSE 0 END) AS inactivos,
        SUM(CASE WHEN dr.resultado_actividad = 'Inconclusive' THEN 1 ELSE 0 END) AS inconclusos,
        SUM(CASE WHEN dr.resultado_actividad NOT IN ('Active', 'Inactive', 'Inconclusive') THEN 1 ELSE 0 END) AS otros,
        COUNT(*) AS total,                            -- Total de ensayos
        ROUND(
            SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS pct_activos,                             -- Porcentaje activos
        ROUND(
            SUM(CASE WHEN dr.resultado_actividad = 'Inactive' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS pct_inactivos                            -- Porcentaje inactivos
    FROM fact f
    JOIN dim_comp dc ON f.llave_compuesto = dc.llave_compuesto
    JOIN dim_res dr ON f.llave_resultado = dr.llave_resultado
    GROUP BY dc.nombre_comun, dc.clasificacion_lipinski
    ORDER BY pct_activos DESC
""").show(truncate=False)